### Step 1: Install Dependencies
First, make sure you have MLflow and scikit-learn installed. You can install them using pip:

In [6]:
#!pip3 install mlflow scikit-learn

### Step 2: Import Libraries
In your Python script, import the necessary libraries:

In [13]:
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from urllib.parse import urlparse

import numpy as np
import pandas as pd
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
mlflow.autolog() 


2025/11/02 18:56:37 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


### Step 3: Load and Prepare Data
Load a dataset from scikit-learn, split it into training and testing sets, and perform any necessary data preprocessing:

In [14]:
data = load_diabetes()
X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Step 4: Train and Log the Model
Train a simple linear regression model using scikit-learn and log it using MLflow:

In [15]:
with mlflow.start_run():
    # Create and train a linear regression model
    lr = LinearRegression()
    lr.fit(X_train, y_train)

    predictions = lr.predict(X_train)
    signature = infer_signature(X_train, predictions)

    tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

    # Model registry does not work with file store
    if tracking_url_type_store != "file":
        # Register the model
        # There are other ways to use the Model Registry, which depends on the use case,
        # please refer to the doc for more information:
        # https://mlflow.org/docs/latest/model-registry.html#api-workflow
        mlflow.sklearn.log_model(
            lr, "model", registered_model_name="RandomForest", signature=signature
        )
    else:
        mlflow.sklearn.log_model(lr, "model", signature=signature)

2025/11/02 18:56:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


This code initializes an MLflow run, trains the model, and logs it with the name "model."

### Step 5: Save the MLflow Artifacts
By default, MLflow logs the model into an artifact store. You can specify the artifact location in your MLflow server configuration or use the default location. You can also set the environment variable MLFLOW_TRACKING_URI to point to your MLflow server.

### Step 6: Serve the Model
To serve the model using MLflow, you can use the mlflow models serve command. First, make sure that MLflow is running as a server (e.g., mlflow server).

Then, you can start serving your model:

### In case of Error

In [10]:
# https://github.com/pyenv/pyenv/wiki#suggested-build-environment

In [11]:
# !rm -rf '/home/ramin/.pyenv'

In [12]:
# !curl https://pyenv.run | bash
# !python -m  pip install virtualenv
# !PATH="$HOME/.pyenv/bin:$PATH"
# !export PYENV_ROOT="$HOME/.pyenv"

### In-case of dependeny error

In [13]:
# mlflow.pyfunc.get_model_dependencies('mlruns/0/e6f6f958a20c407f9d485ba5fb24f6c2/artifacts/model')

In [14]:
# !pip3 install -r mlruns/0/aea73ab36d544af29927cf26199b8888/artifacts/model/requirements.txt

### To run on a severate bash screen within the same venv

In [1]:
!mlflow models serve --env-manager=local -m mlruns/0/models/m-ea6cba408a9e4a5d804c660f599933ef/artifacts -h 127.0.0.1 -p 5001

2025/11/02 19:00:04 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'
2025/11/02 19:00:04 INFO mlflow.pyfunc.backend: === Running command 'exec uvicorn --host 127.0.0.1 --port 5001 --workers 1 mlflow.pyfunc.scoring_server.app:app'
INFO:     Started server process [7740]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:5001 (Press CTRL+C to quit)
^C

Aborted!
INFO:     Shutting down
INFO:     Finished server process [7740]
ERROR:    Traceback (most recent call last):
  File "/opt/anaconda3/envs/mlopslab/lib/python3.14/asyncio/runners.py", line 204, in run
    return runner.run(main)
           ~~~~~~~~~~^^^^^^
  File "/opt/anaconda3/envs/mlopslab/lib/python3.14/asyncio/runners.py", line 127, in run
    return self._loop.run_until_complete(task)
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^
  File "/opt/anaconda3/envs/mlopslab/lib/python3.14/asyncio/base_events.py", li

### Step 7: Make Predictions
You can make predictions by sending HTTP POST requests to the model server. Here's an example using Python's requests library:

In [16]:
X_test.shape

(89, 10)

In [17]:
import requests
import json

url = 'http://127.0.0.1:5001/invocations'

data = {
    "columns": data['feature_names'],
    "instances": X_test.tolist()
}

response = requests.post(url, json=data)
predictions = response.json()

print(predictions)

ConnectionError: HTTPConnectionPool(host='127.0.0.1', port=5001): Max retries exceeded with url: /invocations (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x31063f890>: Failed to establish a new connection: [Errno 61] Connection refused'))

The url variable, in the context of the code provided, represents the Uniform Resource Locator (URL) that is used to specify the location or address of a web resource. In this case, it points to a specific endpoint on a local web server:

* http:// indicates the protocol being used, which is Hypertext Transfer Protocol (HTTP). HTTP is the foundation of data communication on the World Wide Web.

* localhost is a special hostname that typically refers to the current device or computer where the code is running. It is often used to access services running on the same machine.

* 5000 is the port number. Ports are used to differentiate between different services or processes running on the same machine. Port 5000 is a commonly used port for running web servers locally.

* /invocations is the path or endpoint on the web server. In the context of machine learning deployment, this path often represents an API endpoint for making predictions using a deployed machine learning model. In this specific case, it's likely that the web server running on localhost:5000 has an endpoint called /invocations that allows you to send data to the deployed model and receive predictions in response.

So, the url variable is essentially storing the URL of the API endpoint where you can send a POST request with data to obtain predictions from a locally deployed machine learning model.